In [ ]:
import ase
from ase import Atoms
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
from datetime import datetime
from omegaconf import DictConfig, OmegaConf

import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from spec2struct.diffusion.diffusion_cfg import CSPDiffusion
from spec2struct.utils.constants import cdvae_train_num_elements_distribution
from spec2struct.utils.utils import decode
from spec2struct.dataset.datamodule import CrystalDataModule, worker_init_fn
from spec2struct.dataset.dataset import CrystalDataset

In [ ]:
def diffuse(
    data_loader, 
    model,
    n_candidates,
    step_lr, 
    diff_ratio,
):
    all_outputs = {}

    count = 0

    for idx, batch in enumerate(data_loader):
        batch = batch.to('cuda')
        batch_outputs = []
        batch_stacks = []
        count += 1
        
        for i in range(n_candidates):
            # outputs, _ = model.sample(
            #     batch,
            #     step_lr=step_lr,
            #     diff_ratio=diff_ratio,
            #     conditional=True
            # )
            outputs, stack = model.cfg_sample(
                batch,
                step_lr=step_lr,
                diff_ratio=diff_ratio,
                w=0.2
            )

            outputs = {
                'structure_id': batch.structure_id,
                'num_atoms': outputs['num_atoms'].detach().cpu(),
                'atom_types': outputs['atom_types'].detach().cpu(),
                'frac_coords': outputs['frac_coords'].detach().cpu(),
                'lattices': outputs['lattices'].detach().cpu(),
            }
            batch_outputs.append(outputs)
            batch_stacks.append(stack)
        
        all_outputs[idx] = {
            "batch": batch,
            "outputs": batch_outputs,
            "stack": batch_stacks
        }

        # do just one epoch for testing
        # to reconstruct the entire dataset, comment the following line
    
    return all_outputs

In [ ]:
root_path = Path("outputs/250705_080702_2d_dos_cfg_ft_truncated")
save_path = "structures"
file_name = None
batch_size = 64
n_candidates = 1
step_lr = 5e-6
diff_ratio = 1.0

In [ ]:
config_path = root_path / 'hparams.yaml'
config = OmegaConf.load(config_path)

# load checkpoint
ckpt_path = glob(str(root_path / '*.ckpt'))
if len(ckpt_path) == 0:
    raise ValueError("No checkpoint file found.")
elif len(ckpt_path) > 1:
    raise ValueError("Multiple checkpoint files found.")
ckpt_path = ckpt_path[0]

model = CSPDiffusion.load_from_checkpoint(ckpt_path, config=config)
model.to('cuda')

config.datamodule.batch_size.test = batch_size

data_module = CrystalDataModule(config, scaler_path=str(root_path))
data_module.setup(stage="test")
test_loader = data_module.test_dataloader()

In [ ]:
all_outputs = diffuse(
    test_loader,
    model, 
    n_candidates,
    step_lr,
    diff_ratio
)

In [ ]:
torch.save(all_outputs, "all_outputs.pt")

In [ ]:
import json
f = open("data/2d_dos_truncated/train.json")
raw_data = json.load(f)
f.close()

gt_atoms_dict = {
    x['structure_id']: Atoms(
        numbers=x['atomic_numbers'],
        positions=x['positions'],
        cell=x['cell'],
        pbc=True
    )
    for x in raw_data
}

In [ ]:
matched_ids = []

for k, v in all_outputs.items():
    outputs = v['outputs'][0]
    stacks = v['stack'][0]

    structure_ids = outputs['structure_id']
    atom_types = outputs['atom_types']
    num_atoms  = outputs['num_atoms'] 

    counts = num_atoms.tolist()
    atom_types_split = torch.split(atom_types, counts)

    for _id, _comp in zip(structure_ids, atom_types_split):
        gt_comp = sorted(gt_atoms_dict[_id].get_atomic_numbers().tolist())
        gen_comp = sorted(_comp.tolist())

        if gt_comp == gen_comp:
            matched_ids.append(_id)

In [ ]:
import itertools

In [ ]:
trajectories = {}
for k, v in all_outputs.items():
    outputs = v['outputs'][0]
    stacks = v['stack'][0]

    structure_ids = outputs['structure_id']
    atom_types = outputs['atom_types']
    num_atoms  = outputs['num_atoms'] 

    all_atom_types = stacks['atom_types']
    all_frac_coords = stacks['all_frac_coords']
    all_lattices = stacks['all_lattices']
    
    counts = num_atoms.tolist()
    offsets = [0] + list(itertools.accumulate(counts))

    for i in range(len(counts)):
        start, end = offsets[i], offsets[i+1]
        
        # slice out the per-structure trajectory:
        atom_traj   = all_atom_types[:, start:end]        # (1000, nᵢ)
        coord_traj  = all_frac_coords[:, start:end, :]    # (1000, nᵢ, 3)
        lattice_traj= all_lattices[:, i, :, :]            # (1000, 3, 3)
        
        trajectories[structure_ids[i]] = {
            "atom_types"  : atom_traj,
            "frac_coords" : coord_traj,
            "lattice"     : lattice_traj
        }

In [ ]:
for k in tqdm(matched_ids):
    atoms_list = []
    for _types, _coords, _lattice in zip(
        trajectories[k]['atom_types'], 
        trajectories[k]['frac_coords'], 
        trajectories[k]['lattice']
    ):
        atoms = Atoms(
            numbers=_types.tolist(),
            scaled_positions=_coords.cpu().numpy(),
            cell=_lattice.cpu().numpy(),
            pbc=True
        )
        atoms_list.append(atoms)
    
    save_path = Path(f"structures/{k}_traj")
    save_path.mkdir(parents=True, exist_ok=True)
    for i, atoms in enumerate(atoms_list):
        ase.io.write(save_path / f"{i:0{4}}.cif", atoms)

    ase.io.write(f"structures/extxyz/{k}.extxyz", atoms_list)